# Unsupervised Ellipsoid Fitting Algorithm

This notebook extends the hypersphere-based algorithm by replacing each spherical region with an ellipsoid. The motivation is that normal embeddings in DINOv2 feature space are unlikely to form locally isotropic clusters. Instead, neighbourhoods may stretch more strongly along some directions than others. Ellipsoids are therefore able to model local variance more naturally than hyperspheres.

The ellipsoid formulation has several advantages:

- Aligns each region with the natural variance structure of the local KNN neighbourhood.
- Reduces unused empty space compared with hyperspheres, since the boundary can contract along low-variance directions.
- It can reduce unnecessary overlap between neighbouring regions by following the dominant principal axes of the local embedding distribution.
- It is better suited to high-variance categories, where the normal embedding space may contain elongated or anisotropic regions.
- Provides additional interpretability through eigenvalues, eigenvectors, axis ratios, and local region structure.

Several changes were introduced compared with the hypersphere version:

- Growth is variance-scaled rather than uniform. Expansion along each axis is controlled by the relative eigenvalue contribution, so high-variance directions can grow more than low-variance directions.
- Candidate cleaning is weight-based. Instead of immediately removing a point when a candidate ellipsoid overlaps a previous region, the algorithm first reduces that point’s contribution to the ellipsoid  fit. If its weight reaches zero and overlap remains, the point is removed.
- Sparse ellipsoids require additional support. Unlike hyperspheres, ellipsoids fitted from very few points can become geometrically unstable. To address this, the covariance of a small candidate region is blended with covariance information from a previous ellipsoid.
- The current support strategy borrows covariance from the nearest ellipsoid by centre distance. This is a limitation, since the nearest ellipsoid may not be the most geometrically similar. Future work should select support using both spatial proximity and shape similarity.

In [1]:
import os

import torch
import sqlite3
import pandas as pd

import json

from datetime import datetime
from dataclasses import asdict

from pathlib import Path
import sys 

ROOT = Path.cwd().parents[1]
sys.path.append(str(ROOT))

from src.config.paths import EMBEDS_DIR, EXPERIMENTS, RESULTS, DB_PATH, MODELS
MODEL_PATH = MODELS / "dino_adapter_block/20260812_150510"

EMBED_PATH = EMBEDS_DIR / "dino/20260812_150510"
EMBED_NAME = EMBED_PATH.stem

EXPERIMENTS_DIR = EXPERIMENTS / EMBED_NAME / "ellipsoid"
RESULTS_DIR = RESULTS / EMBED_NAME

In [2]:
os.makedirs(EXPERIMENTS_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

In [3]:
cls_tokens = torch.load(EMBED_PATH/"cls.pt", weights_only=False)

In [4]:
conn = sqlite3.connect(DB_PATH)

meta = pd.read_sql_query("SELECT * FROM meta", conn)
categories = pd.read_sql_query("SELECT DISTINCT category FROM meta", conn)["category"].to_list()

conn.close()

In [5]:
%load_ext autoreload
%autoreload 2

from src.algorithims.ellipsoid import EllipsoidFitter, EllipsoidCover, CandidateCleaner, EllipsoidEvaluator
from src.types import ExperimentConfig, AlgorithmResults

metadata = ExperimentConfig(
    K_frac=0.05,
    start_growth=1.2,
    min_growth=1,
    reg=1e-4,
    growth_type="variance_scaled",
    cleaner="shared_axis"
)

fitter = EllipsoidFitter(support_points=5, reg=metadata.reg)
cleaner = CandidateCleaner(min_points=1, fitter=fitter)

cover = EllipsoidCover(fitter=fitter, cleaner=cleaner)

evaluator = EllipsoidEvaluator(reg=metadata.reg)

In [6]:
with open(MODEL_PATH / "metadata.json", "r") as f:
    model_metadata = json.load(f)
    
neg_indices = model_metadata.get("negative_indices", [])

print(model_metadata)
print(model_metadata.keys())

{'notes': ['Trained using normal dino', 'Testing with lr1e-4 due to historical evidence that it worked well before'], 'git_commit': '5724391b0f7e687f9440411c3ece749d5b651caf', 'parameters': {'seed': 42, 'num_workers': 8, 'pin_memory': True, 'samples_per_category': 16, 'batch_size': 240, 'epochs': 20, 'model_dim': 384, 'hidden_dim': 1536, 'mlp_factor': 4, 'dropout': 0.0, 'model_normaliser': 'layer', 'use_residual': False, 'learning_rate': 0.0001, 'weight_decay': 0.0001, 'model_name': 'model.pt'}, 'parent_models': {'dino': None}, 'negative_indices': [43, 285, 287, 470, 219, 1322, 830, 513, 1016, 268, 18, 327, 841, 751, 968, 621, 426, 1582, 284, 1628, 118, 410, 1197, 501, 1262, 449, 1419, 691, 1422, 1475, 1711, 952, 903, 1620, 1278, 1153, 399, 894, 491, 1489, 708, 1226, 3, 796, 381, 1533, 566, 1597, 1109, 485, 73, 738, 526, 942, 1131, 275, 874, 242, 135, 1242, 54, 1048, 181, 1383, 1712, 555, 997, 1158, 1191, 1717, 1510, 1642, 596, 375, 624], 'model_type': 'dino_adapter_block', 'losses': {

In [7]:
time = datetime.now().strftime("%y-%m-%d_%H-%M-%S")
aurocs = {}

for category in categories:
    print("Running", category)
    outputs_dir = EXPERIMENTS_DIR / category / time
    os.makedirs(outputs_dir, exist_ok=True)

    train_mask = meta["split"] == "train"
    train_meta = meta[train_mask]
    test_meta = meta[
        (~train_mask)
        & (~meta.index.isin(neg_indices))
    ]

    train_cat_mask = train_meta["category"] == category

    train_emb = cls_tokens[train_mask]
    cat_emb = train_emb[train_cat_mask]

    collection = cover.run(
        embeds=cat_emb, output_dir=outputs_dir, 
        k_frac=metadata.K_frac, 
        start_growth=metadata.start_growth, min_growth=metadata.min_growth
        )
    
    good_test_cat_mask = (test_meta["category"] == category) & (test_meta["type"] == "good")
    defect_test_cat_mask = (test_meta["category"] == category) & (test_meta["type"] != "good")

    test_emb = cls_tokens[
        (~train_mask)
        & (~meta.index.isin(neg_indices))
    ]
    defect_test_emb = test_emb[defect_test_cat_mask]
    good_test_emb = test_emb[good_test_cat_mask]

    overlaps_df, num_overlaps = evaluator.overlap(embeds=cat_emb, ellipsoids=collection.ellipsoids)
    overlaps_df.to_csv(outputs_dir / f"overlaps.csv", index=False)

    good_any, good_counts = evaluator.inside_any_count(good_test_emb, collection.ellipsoids)
    defect_any, defect_counts = evaluator.inside_any_count(defect_test_emb, collection.ellipsoids)

    evaluation = evaluator.evaluate_detection(good_test_emb, defect_test_emb, collection)
    evaluation.samples.to_csv(outputs_dir / f"results.csv", index=False)

    diagnostics = evaluator.bucket_diagnostics(evaluation.samples)

    aurocs[category] = evaluation.metrics.auroc

    results = AlgorithmResults(
        config=metadata,
        category=category,
        n_shapes=len(collection),
        auroc=evaluation.metrics.auroc,
        normal_inside=int(good_any.sum()),
        defect_inside=int(defect_any.sum())
    )

    with open(outputs_dir / f"metadata.json", "w") as f:
        json.dump(asdict(results), f)

aurocs_df = pd.DataFrame(aurocs.items(), columns=["Category", "AUROC"])

Running bottle
Running cable
Running capsule
Running carpet
Running grid
Running hazelnut
Running leather
Running metal_nut
Running pill
Running screw
Running tile
Running toothbrush
Running transistor
Running wood
Running zipper


In [8]:
aurocs_df

,Category,AUROC
0,bottle,1.000000
1,cable,0.890742
2,capsule,0.913442
3,carpet,0.991573
4,grid,0.988304
5,hazelnut,0.981424
6,leather,1.000000
7,metal_nut,0.951124
8,pill,0.910256
9,screw,0.932978


In [9]:
df_roc_stats = pd.DataFrame({
    "mean": aurocs_df["AUROC"].mean(),
    "median": aurocs_df["AUROC"].median(),
    "std": aurocs_df["AUROC"].std(),
    "min_cat":  aurocs_df["Category"][aurocs_df["AUROC"].idxmin()],
    "min": aurocs_df["AUROC"].min(),
    "max_cat": aurocs_df["Category"][aurocs_df["AUROC"].idxmax()],
    "max": aurocs_df["AUROC"].max()
}, index=[0]).round(3)

df_roc_stats.to_csv(RESULTS_DIR / "ellipsoid_auroc_stats.csv", index=False)

aurocs_df = aurocs_df.round(3)
aurocs_df.to_csv(RESULTS_DIR /"ellipsoid_aurocs.csv", index=False)

df_roc_stats

,mean,median,std,min_cat,min,max_cat,max
0,0.962,0.982,0.04,cable,0.891,bottle,1.0


In [10]:
display(diagnostics["n_points"])
display(diagnostics["eig_ratio"])
display(diagnostics["n_points_by_class"])
display(diagnostics["eig_ratio_aurocs"])

,mean,count
n_points_bucket,,
"(-0.001, 3.0]",0.95935,123
"(3.0, 5.0]",1.00000,8
"(5.0, 10.0]",1.00000,6
"(10.0, 100.0]",1.00000,9


,mean,count
eigval_ratio_bucket,,
"(5.172, 104.24]",0.970588,68
"(104.24, 133.611]",0.800000,5
"(133.611, 136.826]",1.000000,46
"(136.826, 384.029]",0.925926,27


mean  count
n_points_bucket y_true                 
(-0.001, 3.0]   0.0     0.909091     11
                1.0     0.964286    112
(3.0, 5.0]      0.0     1.000000      6
                1.0     1.000000      2
(5.0, 10.0]     0.0     1.000000      6
                1.0          NaN      0
(10.0, 100.0]   0.0     1.000000      9
                1.0          NaN      0

,eigval_ratio_bucket,auroc,count
0,"(5.172, 104.24]",0.988855,68
1,"(104.24, 133.611]",1.000000,5
2,"(133.611, 136.826]",1.000000,46
3,"(136.826, 384.029]",0.980263,27


In [11]:
ellipsoids_df = collection.to_dataframe()

ellipsoids_df

,ellipsoid_id,raw_eig_ratio,reg_eig_ratio,pc95,rank,pc1_ratio,n_points,threshold,support_id,weights_mean,weights_min,n_reduced_weights
0,0,7.830987,7.824110,10,12,0.185453,13,11.073461,NaN,1.0,1.0,0
1,1,11.462382,11.451398,10,11,0.232854,12,10.081099,NaN,1.0,1.0,0
2,2,11.254210,11.245789,9,10,0.250542,11,9.089169,NaN,1.0,1.0,0
3,3,9.266372,9.261383,9,10,0.260496,11,9.089410,NaN,1.0,1.0,0
4,4,8.970608,8.966650,8,9,0.281814,10,8.098794,NaN,1.0,1.0,0
5,5,5.294916,5.293349,8,9,0.205582,10,8.099150,NaN,1.0,1.0,0
6,6,10.493580,10.489342,7,8,0.334100,9,7.110311,NaN,1.0,1.0,0
7,7,11.759942,11.754756,7,8,0.332960,9,7.110303,NaN,1.0,1.0,0
8,8,5.909530,5.908094,6,7,0.269897,8,6.124581,NaN,1.0,1.0,0
9,9,9.922721,9.918881,6,7,0.320028,8,6.124439,NaN,1.0,1.0,0


In [12]:
print(ellipsoids_df["pc1_ratio"].mean())
print(ellipsoids_df["pc1_ratio"].median())
print(ellipsoids_df["pc95"].mean())
print(ellipsoids_df["pc95"].median())

0.4365656664451121
0.4189408316583632
6.733333333333333
7.0


In [13]:
metrics_df = pd.DataFrame([evaluation.metrics.to_dict()])

metrics_df

,auroc,best_threshold,accuracy,good_accuracy,defect_accuracy,mean_good_score,mean_defect_score,score_gap,false_pos,false_neg,n_winning_ellipsoid,max_winning_fraction
0,0.990132,163110.686554,0.965753,0.96875,0.964912,81416.517719,730353.367103,648936.849384,1,4,25,0.315068
